
# NIPA ICT 규제샌드박스 선택 크롤러 (개선결과 분리판)

**변경점**
- 메인 컬럼에 **개선결과**를 **별도 컬럼**으로 추가했습니다.  
- 메타에도 `src_개선결과`, `extracted_columns` 포함.

**기능 유지**
- 임시허가/실증특례 선택 수집, 페이지 자동 넘김
- 상세는 `*/section[3]/form/div/table/tbody/tr/td` 컨테이너로 **범위 제한**
- `<p><strong>헤더</strong></p> + 이어지는 p` 블록 **우선 파싱**
- `작성일/조회수/첨부파일/등록일/다운로드/파일` **노이즈 제거**
- TD 안의 **중복 제목(서비스명)**은 본문에서 제외
- 표/정의목록/라벨패턴/본문최장텍스트(서비스 내용만) **백업**
- **특수문자 제거**(한글/영문/숫자/공백만)


In [ ]:

# !pip install requests beautifulsoup4 pandas


In [1]:

import os, re, time, json, random, logging
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag

try:
    from caas_jupyter_tools import display_dataframe_to_user
except Exception:
    display_dataframe_to_user = None

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("sandbox_nb")

BASE = "https://www.sandbox.or.kr"
LIST_PATH = "/board/designated_case.do"
DETAIL_PATH = "/board/designated_case_edit.do"

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": BASE + "/",
    "Connection": "keep-alive",
}


In [2]:

def robust_get(session: requests.Session, url: str, max_retries: int = 3, **kwargs) -> requests.Response:
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=kwargs.get("timeout", 15), **kwargs)
            if resp.status_code == 200:
                return resp
            else:
                log.warning("GET %s -> HTTP %s", url, resp.status_code)
        except requests.RequestException as e:
            last_err = e
            log.warning("GET %s attempt %d failed: %s", url, attempt, e)
        time.sleep(0.7 * attempt)
    if last_err:
        raise last_err
    raise RuntimeError(f"GET {url} failed after {max_retries} attempts")


In [3]:

def normalize_date(yyyymmdd: str) -> str:
    yyyymmdd = (yyyymmdd or "").strip()
    if re.fullmatch(r"\d{8}", yyyymmdd):
        return f"{yyyymmdd[:4]}-{yyyymmdd[4:6]}-{yyyymmdd[6:]}"
    return yyyymmdd

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def strip_special(s: Any) -> str:
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = re.sub(r"[^0-9A-Za-z가-힣ㄱ-ㅎㅏ-ㅣ\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normkey(s: str) -> str:
    return strip_special(s).replace(" ", "").lower()

NOISE_TOKENS = ["작성일", "조회수", "첨부파일", "등록일", "다운로드", "파일"]
def is_noise_line(txt: str) -> bool:
    t = clean_text(txt)
    if not t:
        return True
    for kw in NOISE_TOKENS:
        if kw in t:
            return True
    return False


In [4]:

def parse_list_page(html: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(html, "html.parser")
    rows: List[Dict[str, Any]] = []
    tbody = soup.select_one("table.table_col.no_active.case tbody")
    if not tbody:
        return rows

    for tr in tbody.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 6:
            continue

        number = clean_text(tds[0].get_text())
        category = clean_text(tds[1].get_text())
        company = clean_text(tds[2].get_text())

        a = tds[3].find("a")
        svc_name = clean_text(a.get_text()) if a else clean_text(tds[3].get_text())
        bdseq = a.get("data-bdseq") if a else None

        reg_date = normalize_date(clean_text(tds[5].get_text()))
        list_td_text = " | ".join([clean_text(td.get_text(" ", strip=True)) for td in tds])

        detail_url = urljoin(BASE, f"{DETAIL_PATH}?bdSeq={bdseq}") if bdseq else None

        rows.append({
            "번호": number,
            "구분": category,
            "기업명": company,
            "서비스명": svc_name,
            "bdSeq": bdseq,
            "detail_url": detail_url,
            "designated_date": reg_date,
            "list_td_text": list_td_text,
        })
    return rows


In [5]:

def find_content_td(soup: BeautifulSoup) -> Optional[Tag]:
    selectors = [
        "body > section:nth-of-type(3) form div table tbody tr td",
        "section form div table tbody tr td",
        "form div table tbody tr td",
        "table tbody tr td",
    ]
    for sel in selectors:
        els = soup.select(sel)
        if els:
            best = max(els, key=lambda e: len(e.get_text(" ", strip=True)))
            if best and clean_text(best.get_text()):
                return best
    return None


In [6]:

TARGET_COLS = ["서비스 내용","특례 내용","부가조건","규제","기대효과","개선결과"]

HEADER_MAP = {
    "서비스내용": "서비스 내용",
    "내용": "서비스 내용",
    "서비스개요": "서비스 내용",
    "개요": "서비스 내용",
    "사업개요": "서비스 내용",
    "사업내용": "서비스 내용",
    "신청내용": "서비스 내용",
    "제안내용": "서비스 내용",
    "특례내용": "특례 내용",
    "규제특례": "특례 내용",
    "부가조건": "부가조건",
    "조건": "부가조건",
    "조건사항": "부가조건",
    "규제": "규제",
    "관련규제": "규제",
    "규제내용": "규제",
    "규제현황": "규제",
    "현행규제": "규제",
    "기대효과": "기대효과",
    "기대되는효과": "기대효과",
    "정책효과": "기대효과",
    "파급효과": "기대효과",
    "개선결과": "개선결과",
    "개선효과": "개선결과",
    "개선사항": "개선결과",
}

def map_header_to_col(header_text: str) -> Optional[str]:
    k = normkey(header_text)
    if k in HEADER_MAP:
        return HEADER_MAP[k]
    if ("서비스" in k and ("내용" in k or "개요" in k)) or k == "내용":
        return "서비스 내용"
    if "특례" in k:
        return "특례 내용"
    if "부가" in k and "조건" in k or k == "조건":
        return "부가조건"
    if "규제" in k:
        return "규제"
    if "기대" in k and "효과" in k or "파급효과" in k:
        return "기대효과"
    if "개선" in k and ("결과" in k or "효과" in k or "사항" in k):
        return "개선결과"
    return None


In [7]:

def parse_strong_blocks_within(root: Tag, service_title: str="") -> List[Tuple[str,str]]:
    # strong 헤더와 그 본문 묶음 리스트
    pairs = []
    title_norm = normkey(service_title)
    for p in root.find_all("p"):
        strong = p.find("strong")
        if not strong:
            continue
        header = clean_text(strong.get_text(" ", strip=True))
        same_p = clean_text(p.get_text(" ", strip=True))
        same_p = same_p.replace(header, "", 1).strip(" :：-–—·•\u00A0")

        parts = []
        if same_p and not is_noise_line(same_p) and normkey(same_p) != title_norm:
            parts.append(same_p)

        nxt = p.find_next_sibling("p")
        while nxt and not nxt.find("strong"):
            t = clean_text(nxt.get_text(" ", strip=True))
            if t and not is_noise_line(t) and normkey(t) != title_norm:
                parts.append(t)
            nxt = nxt.find_next_sibling("p")

        if parts:
            pairs.append((header, clean_text(" ".join(parts))))
    return pairs


In [8]:

DETAIL_KEYS = {
    "서비스 내용": ["서비스 내용","내용","사업내용","서비스내용","신청내용","서비스 개요","개요","사업 개요","제안내용"],
    "특례 내용": ["특례 내용","특례내용","규제특례","규제 특례","실증특례 내용","실증 특례 내용","특례"],
    "부가조건": ["부가조건","부가 조건","조건","조건사항","조건 사항","부가 조건 사항"],
    "규제": ["규제","관련 규제","규제 내용","규제내용","규제현황","현행 규제"],
    "기대효과": ["기대효과","기대 효과","효과","기대되는 효과","정책효과","파급효과"],
    "개선결과": ["개선결과","개선 결과","개선사항","개선 사항","개선효과","개선 효과"],
}

def parse_detail_fields_from_pairs_within(root: Tag) -> Tuple[Dict[str, str], Dict[str, str]]:
    values = {c:"" for c in TARGET_COLS}
    sources = {c:"" for c in TARGET_COLS}

    for table in root.find_all("table"):
        for tr in table.find_all("tr"):
            ths = tr.find_all(["th"])
            tds = tr.find_all(["td"])
            if len(ths) == 1 and len(tds) >= 1:
                k = clean_text(ths[0].get_text(" "))
                v = clean_text(" ".join(td.get_text(" ", strip=True) for td in tds))
                if k and v and not is_noise_line(v):
                    for col, aliases in DETAIL_KEYS.items():
                        if any(a in k for a in aliases):
                            values[col] = v; sources[col] = "pairs"
            elif len(ths) >= 1 and len(ths) == len(tds):
                for th, td in zip(ths, tds):
                    k = clean_text(th.get_text(" "))
                    v = clean_text(td.get_text(" ", strip=True))
                    if k and v and not is_noise_line(v):
                        for col, aliases in DETAIL_KEYS.items():
                            if any(a in k for a in aliases):
                                values[col] = v; sources[col] = "pairs"

    for dl in root.find_all("dl"):
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        if len(dts) == len(dds) and len(dts) > 0:
            for dt, dd in zip(dts, dds):
                k = clean_text(dt.get_text(" "))
                v = clean_text(dd.get_text(" ", strip=True))
                if k and v and not is_noise_line(v):
                    for col, aliases in DETAIL_KEYS.items():
                        if any(a in k for a in aliases):
                            values[col] = v; sources[col] = "pairs"

    return values, sources


In [9]:

def longest_block_text_within(root: Tag) -> str:
    txt = clean_text(root.get_text(" ", strip=True))
    if txt and not is_noise_line(txt):
        return txt
    return ""


In [10]:

def parse_detail_page(html: str, service_title: str="") -> Tuple[Dict[str, str], Dict[str, str], List[Tuple[str,str]]]:
    soup = BeautifulSoup(html, "html.parser")
    content_td = find_content_td(soup) or soup

    strong_pairs_all = parse_strong_blocks_within(content_td, service_title=service_title)

    values = {c:"" for c in TARGET_COLS}
    sources = {c:"" for c in TARGET_COLS}
    for header, body in strong_pairs_all:
        col = map_header_to_col(header)
        if col:
            values[col] = (values[col] + " " + body).strip() if values[col] else body
            sources[col] = "strong"

    v2, s2 = parse_detail_fields_from_pairs_within(content_td)
    for k in TARGET_COLS:
        if not values.get(k) and v2.get(k):
            values[k] = v2[k]; sources[k] = s2.get(k) or "pairs"

    full_text = content_td.get_text("\\n", strip=True)
    for key, aliases in DETAIL_KEYS.items():
        if not values[key]:
            for a in aliases:
                m = re.search(rf"{re.escape(a)}\\s*[:：]\\s*(.+)", full_text)
                if m:
                    val = clean_text(m.group(1))
                    if not is_noise_line(val):
                        values[key] = val; sources[key] = "label"
                        break

    if not values["서비스 내용"]:
        backup = longest_block_text_within(content_td)
        if backup:
            values["서비스 내용"] = backup; sources["서비스 내용"] = sources.get("서비스 내용") or "fallback"

    for k in TARGET_COLS:
        if not values.get(k):
            sources[k] = sources.get(k) or "empty"

    return values, sources, strong_pairs_all


In [11]:

def fetch_index(forms=("FORM_002","FORM_003"), max_pages=None, delay=1.0, keyword=None, since=None, until=None) -> pd.DataFrame:
    session = requests.Session()
    all_rows = []
    for form_code in forms:
        page = 1
        while True:
            params = {"pageNumber": page, "applyFormCode": form_code}
            list_url = urljoin(BASE, LIST_PATH)
            resp = robust_get(session, list_url, params=params)
            html = resp.text

            items = parse_list_page(html)
            if not items:
                break

            for it in items:
                if keyword and keyword.lower() not in (it["기업명"] + " " + it["서비스명"]).lower():
                    continue
                ok = True
                if since:
                    try:
                        ok = ok and (datetime.strptime(it["designated_date"], "%Y-%m-%d") >= datetime.strptime(since, "%Y-%m-%d"))
                    except:
                        pass
                if until:
                    try:
                        ok = ok and (datetime.strptime(it["designated_date"], "%Y-%m-%d") <= datetime.strptime(until, "%Y-%m-%d"))
                    except:
                        pass
                if not ok:
                    continue

                it["form_code"] = form_code
                all_rows.append(it)

            if max_pages and page >= max_pages:
                break
            page += 1
            time.sleep(delay + random.uniform(0, delay/2))

    df = pd.DataFrame(all_rows)
    cols = ["form_code", "bdSeq", "번호", "구분", "기업명", "서비스명", "designated_date", "detail_url", "list_td_text"]
    df = df[cols] if set(cols).issubset(df.columns) else df
    return df



## 1) 인덱스 로드


In [12]:

forms = ("FORM_002","FORM_003")
max_pages = None
keyword = None
since = None
until = None

df_index = fetch_index(forms=forms, max_pages=max_pages, keyword=keyword, since=since, until=until)
print(f"Loaded index rows: {len(df_index)}")
print(df_index.groupby("form_code")["bdSeq"].count())

display(df_index.head(20))
if display_dataframe_to_user is not None:
    try:
        display_dataframe_to_user("승인사례 인덱스 (XPath 텍스트 포함)", df_index)
    except Exception as e:
        log.info("Display helper not available: %s", e)


Loaded index rows: 307
form_code
FORM_002     79
FORM_003    228
Name: bdSeq, dtype: int64


,form_code,bdSeq,번호,구분,기업명,서비스명,designated_date,detail_url,list_td_text
0,FORM_002,1120,71,임시허가,(사)한국통신사업자연합회,마이데이터 기반 통신요금 정보제공서비스 개발((사)한국통신사업자연합회)),2024-12-18,https://www.sandbox.or.kr/board/designated_cas...,71 | 임시허가 | (사)한국통신사업자연합회 | 마이데이터 기반 통신요금 정보제공...
1,FORM_002,1120,71,임시허가,(사)한국통신사업자연합회,마이데이터 기반 통신요금 정보제공서비스 개발((사)한국통신사업자연합회)),2024-12-18,https://www.sandbox.or.kr/board/designated_cas...,71 | 임시허가 | (사)한국통신사업자연합회 | 마이데이터 기반 통신요금 정보제공...
2,FORM_002,1040,70,임시허가,가람기획사,이동형 가상현실(VR) 체험 버스(가람기획사),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,70 | 임시허가 | 가람기획사 | 이동형 가상현실(VR) 체험 버스(가람기획사) ...
3,FORM_002,1041,69,임시허가,사단법인 탑교육문화원,이동형 가상현실(VR) 체험 버스(사단법인 탑교육문화원),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,69 | 임시허가 | 사단법인 탑교육문화원 | 이동형 가상현실(VR) 체험 버스(사...
4,FORM_002,1042,68,임시허가,주식회사 버터플라이드림,이동형 가상현실(VR) 체험 트럭(주식회사 버터플라이드림),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,68 | 임시허가 | 주식회사 버터플라이드림 | 이동형 가상현실(VR) 체험 트럭(...
5,FORM_002,1043,67,임시허가,(주)브이리스브이알,이동형 가상현실(VR) 체험 트럭((주)브이리스브이알),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,67 | 임시허가 | (주)브이리스브이알 | 이동형 가상현실(VR) 체험 트럭((주...
6,FORM_002,1044,66,임시허가,투어이즈,이동형 가상현실(VR) 체험 트럭(투어이즈),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,66 | 임시허가 | 투어이즈 | 이동형 가상현실(VR) 체험 트럭(투어이즈) | ...
7,FORM_002,1045,65,임시허가,주식회사 디지티모빌리티,플랫폼 기반 임시 택시운전 자격 운영(주식회사 디지티모빌리티),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,65 | 임시허가 | 주식회사 디지티모빌리티 | 플랫폼 기반 임시 택시운전 자격 운...
8,FORM_002,1046,64,임시허가,진모빌리티,플랫폼 기반 임시 택시운전 자격 운영(진모빌리티),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,64 | 임시허가 | 진모빌리티 | 플랫폼 기반 임시 택시운전 자격 운영(진모빌리티...
9,FORM_002,1047,63,임시허가,KM솔루션,플랫폼 기반 임시 택시운전 자격 운영(KM솔루션),2024-06-28,https://www.sandbox.or.kr/board/designated_cas...,63 | 임시허가 | KM솔루션 | 플랫폼 기반 임시 택시운전 자격 운영(KM솔루션...



## 2) 선택하기


In [13]:

selected_bdseqs = "ALL"
if selected_bdseqs == "ALL":
    selected_bdseqs = df_index["bdSeq"].dropna().astype(str).unique().tolist()
else:
    selected_bdseqs = [str(x) for x in selected_bdseqs]
print("Selected size:", len(selected_bdseqs))


Selected size: 278



## 3) 상세 크롤링 + 저장 (10컬럼 본문 + 메타)
- 메인: 10개 컬럼(**개선결과 분리, 특수문자 제거 적용**)
- 메타: 어떤 컬럼을 **어떤 소스**로 채웠는지 + `list_td_text`


In [19]:

def crawl_details_for_selected(df_index: pd.DataFrame, selected_bdseqs: List[str],
                               outdir: str = "./out", delay: float = 0.8,
                               sanitize_text: bool = True,
                               save_meta: bool = True) -> pd.DataFrame:
    os.makedirs(outdir, exist_ok=True)
    session = requests.Session()
     # --- 여기 추가 ---
    if selected_bdseqs is None or (isinstance(selected_bdseqs, str) and selected_bdseqs.strip().upper() == "ALL"):
        selected_bdseqs = df_index["bdSeq"].dropna().astype(str).unique().tolist()
    else:
        # 리스트/튜플 등으로 넘어온 경우도 문자열화
        selected_bdseqs = [str(x) for x in selected_bdseqs if pd.notna(x)]
    # -----------------
    rows_out, meta_rows = [], []

    for bd in selected_bdseqs:
        row = df_index[df_index["bdSeq"].astype(str) == str(bd)]
        if row.empty:
            log.warning("bdSeq %s not in df_index; skip", bd)
            continue
        r = row.iloc[0].to_dict()
        detail_url = r.get("detail_url")
        try:
            if detail_url:
                dresp = robust_get(session, detail_url)
                values, sources, strong_all = parse_detail_page(dresp.text, service_title=r.get("서비스명",""))
            else:
                values = {c:"" for c in TARGET_COLS}
                sources = {k:"empty" for k in TARGET_COLS}
        except Exception as e:
            log.warning("Detail fetch failed (bdSeq=%s): %s", bd, e)
            values = {c:"" for c in TARGET_COLS}
            sources = {k:"empty" for k in TARGET_COLS}

        rec = {
            "번호": r.get("번호",""),
            "구분": r.get("구분",""),
            "기업명": r.get("기업명",""),
            "서비스명": r.get("서비스명",""),
            "서비스 내용": values.get("서비스 내용",""),
            "특례 내용": values.get("특례 내용",""),
            "부가조건": values.get("부가조건",""),
            "규제": values.get("규제",""),
            "기대효과": values.get("기대효과",""),
            "개선결과": values.get("개선결과",""),
        }
        if sanitize_text:
            rec = {k: strip_special(v) for k, v in rec.items()}
        rows_out.append(rec)

        if save_meta:
            filled_cols = [k for k in TARGET_COLS if rec.get(k)]
            meta_row = {
                "bdSeq": r.get("bdSeq",""),
                "form_code": r.get("form_code",""),
                "designated_date": r.get("designated_date",""),
                "detail_url": r.get("detail_url",""),
                "list_td_text": strip_special(r.get("list_td_text","")) if sanitize_text else r.get("list_td_text",""),
                "extracted_columns": "|".join(filled_cols),
                "src_서비스 내용": sources.get("서비스 내용",""),
                "src_특례 내용": sources.get("특례 내용",""),
                "src_부가조건": sources.get("부가조건",""),
                "src_규제": sources.get("규제",""),
                "src_기대효과": sources.get("기대효과",""),
                "src_개선결과": sources.get("개선결과",""),
            }
            meta_rows.append(meta_row)

        time.sleep(delay + random.uniform(0, delay/2))

    df = pd.DataFrame(rows_out, columns=[
        "번호","구분","기업명","서비스명","서비스 내용","특례 내용","부가조건","규제","기대효과","개선결과"
    ])

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(outdir, exist_ok=True)
    csv_path = os.path.join(outdir, f"sandbox_selected_{ts}.csv")
    jsonl_path = os.path.join(outdir, f"sandbox_selected_{ts}.jsonl")
    if not df.empty:
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for _, row in df.iterrows():
                f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\\n")
    print("Saved MAIN CSV ->", csv_path)
    print("Saved MAIN JSONL ->", jsonl_path)

    if save_meta and meta_rows:
        dfm = pd.DataFrame(meta_rows, columns=[
            "bdSeq","form_code","designated_date","detail_url","list_td_text","extracted_columns",
            "src_서비스 내용","src_특례 내용","src_부가조건","src_규제","src_기대효과","src_개선결과"
        ])
        meta_csv = os.path.join(outdir, f"sandbox_selected_meta_{ts}.csv")
        meta_jsonl = os.path.join(outdir, f"sandbox_selected_meta_{ts}.jsonl")
        dfm.to_csv(meta_csv, index=False, encoding="utf-8-sig")
        with open(meta_jsonl, "w", encoding="utf-8") as f:
            for _, row in dfm.iterrows():
                f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\\n")
        print("Saved META CSV  ->", meta_csv)
        print("Saved META JSONL->", meta_jsonl)

    return df


In [20]:

# 기본 실행 스텁 (원하는 경우 바로 실행)
OUTDIR = "./out"
df_selected = crawl_details_for_selected(
    df_index=df_index,
    selected_bdseqs="ALL",  # 또는 ["1120","1045"]
    outdir=OUTDIR,
    delay=0.8,
    sanitize_text=True,
    save_meta=True
)
print(f"Crawled detail rows: {len(df_selected)}")
display(df_selected.head(10))


Saved MAIN CSV -> ./out/sandbox_selected_20251024_070511.csv
Saved MAIN JSONL -> ./out/sandbox_selected_20251024_070511.jsonl
Saved META CSV  -> ./out/sandbox_selected_meta_20251024_070511.csv
Saved META JSONL-> ./out/sandbox_selected_meta_20251024_070511.jsonl
Crawled detail rows: 278


,번호,구분,기업명,서비스명,서비스 내용,특례 내용,부가조건,규제,기대효과,개선결과
0,71,임시허가,사 한국통신사업자연합회,마이데이터 기반 통신요금 정보제공서비스 개발 사 한국통신사업자연합회,통신사로부터 이용자의 실제 통신정보 마이데이터 를 전송받고 이를 기반으로 이용자 요...,한국통신사업자연합회가 개인정보관리 전문기관으로서의 지위를 갖고 마이데이터 기반 통신...,ㅇ 개인정보보호법 시행령 개정을 통해 전송요구권 관련 세부 절차 마련 시 제도 시행...,,,
1,70,임시허가,가람기획사,이동형 가상현실 VR 체험 버스 가람기획사,이동형 VR 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정부 지자체에서 주최...,가람기획사가 VR 체험 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정부 지자...,1 이용자 보호를 위해 안전요원 배치 책임보험 가입 안내판 설치 주의사항 및 이용자...,자동차 튜닝시 교통안전공단 승인을 거치도록 하고 있으나 현재 VR 차량 관련 규정은...,VR 수요지역 지자체 대학교 등 에 이동하여 체험서비스 제공 추진 VR 콘텐츠 이용...,
2,69,임시허가,사단법인 탑교육문화원,이동형 가상현실 VR 체험 버스 사단법인 탑교육문화원,이동형 VR 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정부 지자체에서 주최...,사단법인 탑교육문화원 VR 체험 차량을 활용하여 학교 및 공공기관이 주최하는 행사 ...,1 이용자 보호를 위해 안전요원 배치 책임보험 가입 안내판 설치 주의사항 및 이용자...,자동차 튜닝시 교통안전공단 승인을 거치도록 하고 있으나 현재 VR 차량 관련 규정은...,VR 수요지역 지자체 대학교 등 에 이동하여 체험서비스 제공 추진 VR 콘텐츠 이용...,
3,68,임시허가,주식회사 버터플라이드림,이동형 가상현실 VR 체험 트럭 주식회사 버터플라이드림,이동형 VR 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정부 지자체에서 주최...,주식회사 버터플라이드림이 VR 체험 차량을 활용하여 학교 및 공공기관이 주최하는 행...,1 이용자 보호를 위해 안전요원 배치 책임보험 가입 안내판 설치 주의사항 및 이용자...,자동차 튜닝시 교통안전공단 승인을 거치도록 하고 있으나 현재 VR 차량 트럭 관련 ...,VR 수요지역 지자체 대학교 등 에 이동하여 체험서비스 제공 추진 VR 콘텐츠 이용...,
4,67,임시허가,주 브이리스브이알,이동형 가상현실 VR 체험 트럭 주 브이리스브이알,이동형 VR 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정부 지자체에서 주최...,주 브이리스브이알이 VR 체험 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정...,1 이용자 보호를 위해 안전요원 배치 책임보험 가입 안내판 설치 주의사항 및 이용자...,자동차 튜닝시 교통안전공단 승인을 거치도록 하고 있으나 현재 VR 차량 트럭 관련 ...,VR 수요지역 지자체 대학교 등 에 이동하여 체험서비스 제공 추진 VR 콘텐츠 이용...,
5,66,임시허가,투어이즈,이동형 가상현실 VR 체험 트럭 투어이즈,이동형 VR 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정부 지자체에서 주최...,투어이즈가 VR 체험 차량을 활용하여 학교 및 공공기관이 주최하는 행사 정부 지자체...,1 이용자 보호를 위해 안전요원 배치 책임보험 가입 안내판 설치 주의사항 및 이용자...,자동차 튜닝시 교통안전공단 승인을 거치도록 하고 있으나 현재 VR 차량 트럭 관련 ...,VR 수요지역 지자체 대학교 등 에 이동하여 체험서비스 제공 추진 VR 콘텐츠 이용...,
6,65,임시허가,주식회사 디지티모빌리티,플랫폼 기반 임시 택시운전 자격 운영 주식회사 디지티모빌리티,택시 운전업무에 종사하려는 자 이하 구직자 가 택시 운전자격 취득 전 임시로 플랫폼...,디지티모빌리티가 플랫폼 기반 임시 택시운전 자격 운영 에 대해 실증할 수 있도록 2...,1 금회 임시허가를 받은 사업자 이하 사업자 는 한국교통안전공단 이하 공단 운수종사...,여객자동차법상 택시 운전업무에 종사하려는 자는 택시운전 자격 취득과 법정필수교육을 ...,구직자에게 빠른 일자리를 제공하여 택시업계의 구인난 해소 등에 기여할 수 있을 것으...,
7,64,임시허가,진모빌리티,플랫폼 기반 임시 택시운전 자격 운영 진모빌리티,택시 운전업무에 종사하려는 자 이하 구직자 가 택시 운전자격 취득 전 임시로 플랫폼...,진모빌리티가 플랫폼 기반 임시 택시운전 자격 운영 에 대해 실증할 수 있도록 2년간...,1 금회 임시허가를 받은 사업자 이하 사업자 는 한국교통안전공단 이하 공단 운수종사...,여객자동차법상 택시 운전업무에 종사하려는 자는 택시운전 자격 취득과 법정필수교육을 ...,구직자에게 빠른 일자리를 제공하여 택시업계의 구인난 해소 등에 기여할 수 있을 것으...,
8,63,임시허가,KM솔루션,플랫폼 기반 임시 택시운전 자격 운영 KM솔루션,택시 운전업무에 종사하려는 자 이하 구직자 가 택시 운전자격 취득 전 임시로 플랫폼...,KM솔루션이 플랫폼 기반 임시 택시운전 자격 운영 에 대해 실증할 수 있도록 2년간...,1 금회 임시허가를 받은 사업자 이하 사업자 는 한국교통안전공단 이하 공단 운수종사...,여객자동차법상 택시 운전업무에 종사하려는 자는 택시운전 자격 취득과 법정필수교육을 ...,구직자에게 빠른 일자리를 제공하여 택시업계의 구인난 해소 등에 기여할 수 있을 것으...,
9,62,임시허가,코액터스 주식회사,플랫폼 기반 임시 택시운전 자격 운영 코액터스 주식회사,택시 운전업무에 종사하려는 자 이하 구직자 가 택시 운전자격 취득 전 임시로 플랫폼...,코액터스가 플랫폼 기반 임시 택시운전 자격 운영 에 대해 실증할 수 있도록 2년간 ...,1 금회 임시허가를 받은 사업자 이하 사업자 는 한국교통안전공단 이하 공단 운수종사...,여객자동차법상 택시 운전업무에 종사하려는 자는 택시운전 자격 취득과 법정필수교육을 ...,구직자에게 빠른 일자리를 제공하여 택시업계의 구인난 해소 등에 기여할 수 있을 것으...,


In [23]:
df_selected["개선결과"]

0       
1       
2       
3       
4       
      ..
273     
274     
275     
276     
277     
Name: 개선결과, Length: 278, dtype: object